# 5.3 强化学习 (Reinforcement Learning)

> **课时**: 约2课时 | **难度**: ⭐⭐⭐⭐ | **标签**: `强化学习` `MDP` `Q-Learning` `迷宫`

---

## 📚 本节目标

- 理解强化学习的基本框架：Agent、Environment、State、Action、Reward
- 理解马尔可夫决策过程（MDP）的数学描述
- 理解探索与利用的权衡（ε-greedy 策略）
- 掌握 Q-Learning 算法原理
- 实现简单的 Q-Learning 走迷宫


## 一、强化学习基本概念

### 1.1 什么是强化学习？

强化学习是一种通过**与环境的交互**来学习最优行为策略的方法。智能体通过尝试不同的动作，根据环境反馈的奖励信号来调整自己的行为。

与监督学习的关键区别：

| | 监督学习 | 强化学习 |
|---|---|---|
| 反馈 | 每一步都有正确答案 | 只有奖励信号（延迟反馈） |
| 数据 | 静态数据集 | 动态与环境交互 |
| 目标 | 预测标签 | 最大化累积奖励 |
| 标签 | ✅ 有标签 | ❌ 无标签，只有奖励 |

### 1.2 核心要素

强化学习系统的五个核心要素：

```
    ┌──────────────────────────────────────────┐
    │           Environment (环境)              │
    │                                          │
    │   State sₜ ──▶ Agent ──▶ Action aₜ      │
    │       ◀──────────────────┘              │
    │                                          │
    │            Reward rₜ₊₁                   │
    │            State sₜ₊₁                    │
    └──────────────────────────────────────────┘

    t = 0, 1, 2, 3, ... (时间步)
```

| 要素 | 符号 | 说明 | 示例 |
|------|------|------|------|
| **智能体 (Agent)** | — | 学习和做决策的主体 | 走迷宫的机器人 |
| **环境 (Environment)** | — | Agent 所处的世界 | 迷宫地图 |
| **状态 (State)** | $s_t$ | 环境当前的情况 | 机器人在迷宫中的位置 |
| **动作 (Action)** | $a_t$ | Agent 可以采取的行为 | 上/下/左/右 |
| **奖励 (Reward)** | $r_{t+1}$ | 环境对动作的反馈 | 走到目标+100，碰墙-1 |

### 1.3 目标：最大化累积奖励

智能体的目标是选择一个策略 $\pi$，使得**累积奖励**最大：

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

其中 $\gamma \in [0, 1]$ 是**折扣因子 (Discount Factor)**:

- $\gamma$ 接近 1: 重视长远奖励（有远见）
- $\gamma$ 接近 0: 只看眼前奖励（短视）

用递归形式表示（Bellman方程）:

$$G_t = r_{t+1} + \gamma G_{t+1}$$

## 二、马尔可夫决策过程（MDP）

### 2.1 MDP 定义

马尔可夫决策过程（Markov Decision Process）用五元组 $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$ 来描述强化学习问题：

| 元素 | 符号 | 说明 |
|------|------|------|
| 状态空间 | $\mathcal{S}$ | 所有可能的状态集合 |
| 动作空间 | $\mathcal{A}$ | 所有可能的动作集合 |
| 状态转移概率 | $P(s' | s, a)$ | 在状态 $s$ 执行动作 $a$ 后转移到 $s'$ 的概率 |
| 奖励函数 | $R(s, a, s')$ | 执行动作后获得的即时奖励 |
| 折扣因子 | $\gamma$ | 未来奖励的衰减系数 |

### 2.2 马尔可夫性质

"未来只与现在有关，与过去无关":

$$P(s_{t+1} | s_0, a_0, s_1, a_1, \ldots, s_t, a_t) = P(s_{t+1} | s_t, a_t)$$

### 2.3 策略 (Policy)

- **确定性策略**: $\pi(s) = a$ （给定状态，确定地选择一个动作）
- **随机性策略**: $\pi(a|s) = P(a_t = a | s_t = s)$ （给定状态，以概率选择动作）

### 2.4 状态价值函数 & 动作价值函数

**状态价值函数** $V^\pi(s)$: 从状态 $s$ 开始，遵循策略 $\pi$ 的期望累积奖励：

$$V^\pi(s) = \mathbb{E}_\pi \left[ G_t \mid s_t = s \right]$$

**动作价值函数** $Q^\pi(s, a)$: 在状态 $s$ 执行动作 $a$ 后，再遵循策略 $\pi$ 的期望累积奖励：

$$Q^\pi(s, a) = \mathbb{E}_\pi \left[ G_t \mid s_t = s, a_t = a \right]$$

**最优动作价值函数**:

$$Q^*(s, a) = \max_\pi Q^\pi(s, a)$$

如果能找到 $Q^*(s, a)$，最优策略就是: $\pi^*(s) = \arg\max_a Q^*(s, a)$

## 三、探索与利用的权衡

### 3.1 两难困境

- **利用 (Exploitation)**: 选择当前已知最好的动作
  - 优点: 获得已知的高回报
  - 缺点: 可能错过更好的选择

- **探索 (Exploration)**: 尝试未知动作以获取更多信息
  - 优点: 可能发现更优策略
  - 缺点: 短期可能获得低回报

```
例子: 你在一家餐厅用餐

  利用 → 继续点你最喜欢的菜（确定好吃）
  探索 → 尝试新菜品（可能更好吃，也可能踩雷）

  最优策略: 大部分时候利用，偶尔探索！
```

### 3.2 ε-greedy 策略

最常用的探索策略：

$$a_t = \begin{cases} \arg\max_a Q(s_t, a) & \text{概率 } 1 - \epsilon \text{ (利用)} \\ \text{随机选择} & \text{概率 } \epsilon \text{ (探索)} \end{cases}$$

**ε 衰减策略**: 训练初期 ε 较大（多探索），后期 ε 减小（多利用）：

$$\epsilon_t = \max(\epsilon_{min}, \epsilon_0 \cdot \text{decay}^t)$$

```
ε = 1.0  ──┐
            │ 探索为主
ε = 0.5  ──┤
            │ 探索 + 利用
ε = 0.1  ──┤
            │ 利用为主
ε → 0    ──┘  几乎纯利用
            ───────────▶ 训练进度
```

In [1]:
# ε-greedy 策略实现
import numpy as np
import random

def epsilon_greedy(q_values, epsilon):
    """ε-greedy 策略选择动作
    
    Args:
        q_values: 当前状态下各动作的Q值 [Q(a_0), Q(a_1), ..., Q(a_n)]
        epsilon: 探索概率
    Returns:
        选择的动作索引
    """
    if random.random() < epsilon:
        return random.randint(0, len(q_values) - 1)  # 探索
    else:
        return np.argmax(q_values)  # 利用

# 测试 ε-greedy
q_values = np.array([1.2, 3.5, 0.8, 2.1])  # 动作1的Q值最大
action_names = ['上', '右', '下', '左']

print(f"Q值: {dict(zip(action_names, q_values))}")
print(f"\n不同 ε 值下的选择分布 (1000次试验):")
for eps in [0.0, 0.1, 0.3, 0.5, 1.0]:
    actions = [epsilon_greedy(q_values, eps) for _ in range(1000)]
    counts = [actions.count(i) for i in range(4)]
    print(f"  ε={eps:.1f}: {dict(zip(action_names, counts))}")

Q值: {'上': 1.2, '右': 3.5, '下': 0.8, '左': 2.1}

不同 ε 值下的选择分布 (1000次试验):
  ε=0.0: {'上': 0, '右': 1000, '下': 0, '左': 0}
  ε=0.1: {'上': 25, '右': 912, '下': 27, '左': 36}
  ε=0.3: {'上': 78, '右': 712, '下': 65, '左': 145}
  ε=0.5: {'上': 128, '右': 493, '下': 126, '左': 253}
  ε=1.0: {'上': 258, '右': 232, '下': 264, '左': 246}


## 四、Q-Learning 算法

### 4.1 核心思想

Q-Learning 是一种**无模型 (Model-Free)** 的强化学习算法，直接学习最优动作价值函数 $Q^*(s, a)$，不需要知道环境的状态转移概率。

### 4.2 Bellman 最优方程

$$Q^*(s, a) = R(s, a) + \gamma \max_{a'} Q^*(s', a')$$

### 4.3 Q-Learning 更新规则

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ \underbrace{r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')}_{\text{TD 目标}} - \underbrace{Q(s_t, a_t)}_{\text{当前估计}} \right]$$

其中：
- $\alpha$: 学习率（控制更新幅度）
- $\gamma$: 折扣因子（控制未来奖励的重要性）
- $r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')$: TD 目标（时序差分目标）

### 4.4 完整算法流程

```
算法: Q-Learning

初始化 Q(s, a) = 0  对所有 s, a

for episode = 1 to M:
    初始化状态 s
    for t = 1 to T:
        用 ε-greedy 从 Q(s, ·) 选择动作 a
        执行动作 a，观察奖励 r 和新状态 s'
        Q(s, a) ← Q(s, a) + α[r + γ max_a' Q(s', a') - Q(s, a)]
        s ← s'
        if s 是终止状态: break
    衰减 ε
```

**关键**: Q-Learning 是 **off-policy** 方法，因为它直接逼近 $Q^*$，而不需要遵循某个特定的策略来采集数据。

## 五、实战：Q-Learning 走迷宫

现在让我们实现一个完整的 Q-Learning 来解决走迷宫问题。

### 5.1 迷宫环境定义

In [2]:
# ============================================
# 定义迷宫环境
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import random

class MazeEnvironment:
    """简单的网格迷宫环境"""
    
    # 动作: 0=上, 1=右, 2=下, 3=左
    ACTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]  # (col_delta, row_delta)
    ACTION_NAMES = ['上', '右', '下', '左']
    
    def __init__(self):
        # 迷宫布局: 0=空地, 1=墙壁, 2=终点
        self.maze = np.array([
            [0, 0, 0, 1, 0, 0, 0, 0],
            [1, 1, 0, 1, 0, 1, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 1, 1, 1, 0, 1, 0],
            [0, 0, 0, 0, 1, 0, 1, 0],
            [1, 1, 0, 1, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 1, 1, 0],
            [0, 1, 0, 0, 0, 0, 2, 0],
        ])
        self.rows, self.cols = self.maze.shape
        self.start = (0, 0)  # 起点
        self.goal = np.argwhere(self.maze == 2)[0]  # 终点
        self.n_actions = 4

    def reset(self):
        """重置环境到初始状态"""
        self.agent_pos = self.start
        return self.start

    def step(self, action):
        """执行动作，返回 (新状态, 奖励, 是否结束)"""
        row, col = self.agent_pos
        dc, dr = self.ACTIONS[action]
        new_row, new_col = row + dr, col + dc
        
        # 检查边界和墙壁
        if (0 <= new_row < self.rows and 0 <= new_col < self.cols 
            and self.maze[new_row, new_col] != 1):
            self.agent_pos = (new_row, new_col)
        else:
            # 撞墙：位置不变，小惩罚
            return self.agent_pos, -2.0, False
        
        # 检查是否到达终点
        if self.agent_pos == tuple(self.goal):
            return self.agent_pos, 100.0, True
        
        # 普通移动：小惩罚（鼓励快速到达）
        return self.agent_pos, -0.1, False

    def get_n_states(self):
        """返回可达状态数（用于Q表大小）"""
        return self.rows * self.cols

    def state_to_idx(self, state):
        """将 (row, col) 转换为索引"""
        return state[0] * self.cols + state[1]

# 创建环境
env = MazeEnvironment()
print(f"迷宫大小: {env.rows} x {env.cols}")
print(f"起点: {env.start}")
print(f"终点: {tuple(env.goal)}")
print(f"可达状态数: {env.get_n_states()}")
print(f"动作数: {env.n_actions} ({env.ACTION_NAMES})")

迷宫大小: 8 x 8
起点: (0, 0)
终点: (7, 6)
可达状态数: 64
动作数: 4 (['上', '右', '下', '左'])


In [3]:
# 可视化迷宫
def draw_maze(env, agent_pos=None, path=None, ax=None):
    """绘制迷宫"""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    # 绘制网格
    for r in range(env.rows):
        for c in range(env.cols):
            if env.maze[r, c] == 1:
                ax.add_patch(plt.Rectangle((c, env.rows-1-r), 1, 1,
                             facecolor='#2c3e50', edgecolor='#ecf0f1', linewidth=1))
            else:
                color = '#ecf0f1'
                ax.add_patch(plt.Rectangle((c, env.rows-1-r), 1, 1,
                             facecolor=color, edgecolor='#bdc3c7', linewidth=1))
    
    # 终点
    gr, gc = env.goal
    ax.add_patch(plt.Rectangle((gc, env.rows-1-gr), 1, 1,
                 facecolor='#2ecc71', edgecolor='#27ae60', linewidth=2))
    ax.text(gc+0.5, env.rows-1-gr+0.5, 'GOAL', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    
    # 起点
    ax.add_patch(plt.Rectangle((0, env.rows-1), 1, 1,
                 facecolor='#3498db', edgecolor='#2980b9', linewidth=2))
    ax.text(0.5, env.rows-1+0.5, 'S', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
    
    # 绘制路径
    if path and len(path) > 1:
        for i in range(len(path)-1):
            r1, c1 = path[i]
            r2, c2 = path[i+1]
            ax.plot([c1+0.5, c2+0.5], [env.rows-1-r1+0.5, env.rows-1-r2+0.5],
                    'r-', linewidth=2, alpha=0.7)
    
    # 绘制智能体
    if agent_pos:
        ar, ac = agent_pos
        ax.plot(ac+0.5, env.rows-1-ar+0.5, 'o', markersize=15,
                color='#e74c3c', markeredgecolor='white', markeredgewidth=2)
    
    ax.set_xlim(-0.1, env.cols+0.1)
    ax.set_ylim(-0.1, env.rows+0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    
    return ax

fig, ax = plt.subplots(figsize=(7, 7))
draw_maze(env, ax=ax)
ax.set_title('Maze Environment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('maze_environment.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 700x700 with 1 Axes>

### 5.2 实现 Q-Learning 算法

In [4]:
# ============================================
# Q-Learning 算法实现
# ============================================

def q_learning_train(env, episodes=500, alpha=0.1, gamma=0.95,
                     epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """
    Q-Learning 训练函数
    
    Args:
        env: 迷宫环境
        episodes: 训练回合数
        alpha: 学习率
        gamma: 折扣因子
        epsilon_start: 初始探索率
        epsilon_end: 最终探索率
        epsilon_decay: 探索率衰减系数
    Returns:
        Q表, 训练记录
    """
    n_states = env.get_n_states()
    n_actions = env.n_actions
    
    # 初始化 Q 表
    Q = np.zeros((n_states, n_actions))
    
    epsilon = epsilon_start
    episode_rewards = []    # 每个回合的总奖励
    episode_steps = []      # 每个回合的步数
    
    for ep in range(episodes):
        state = env.reset()
        state_idx = env.state_to_idx(state)
        total_reward = 0
        steps = 0
        
        while True:
            # ε-greedy 选择动作
            if random.random() < epsilon:
                action = random.randint(0, n_actions - 1)
            else:
                action = np.argmax(Q[state_idx])
            
            # 执行动作
            new_state, reward, done = env.step(action)
            new_state_idx = env.state_to_idx(new_state)
            
            # Q-Learning 更新
            td_target = reward + gamma * np.max(Q[new_state_idx])
            td_error = td_target - Q[state_idx, action]
            Q[state_idx, action] += alpha * td_error
            
            total_reward += reward
            steps += 1
            state = new_state
            state_idx = new_state_idx
            
            if done or steps > 200:  # 防止无限循环
                break
        
        episode_rewards.append(total_reward)
        episode_steps.append(steps)
        
        # 衰减 ε
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        
        if (ep + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])
            avg_steps = np.mean(episode_steps[-100:])
            print(f"Episode {ep+1:4d}: 平均奖励 = {avg_reward:7.2f}, "
                  f"平均步数 = {avg_steps:5.1f}, ε = {epsilon:.4f}")
    
    return Q, episode_rewards, episode_steps

# 训练
print("开始 Q-Learning 训练...\n")
Q_table, rewards, steps = q_learning_train(
    env, episodes=500, alpha=0.1, gamma=0.95,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995
)
print(f"\n训练完成！最终平均奖励: {np.mean(rewards[-100:]):.2f}")

开始 Q-Learning 训练...

Episode  100: 平均奖励 = -33.57, 平均步数 = 46.4, ε = 0.6053
Episode  200: 平均奖励 =  8.34, 平均步数 = 27.0, ε = 0.3660
Episode  300: 平均奖励 =  42.06, 平均步数 = 18.4, ε = 0.2213
Episode  400: 平均奖励 =  58.12, 平均步数 = 14.2, ε = 0.1339
Episode  500: 平均奖励 =  70.38, 平均步数 = 12.0, ε = 0.0811

训练完成！最终平均奖励: 70.38


In [5]:
# 可视化训练过程
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 左图：每回合奖励（带滑动平均）
axes[0].plot(rewards, alpha=0.3, color='#3498db', label='Raw Reward')
# 50回合滑动平均
window = 50
if len(rewards) >= window:
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards)), moving_avg, 'r-', linewidth=2,
                 label=f'{window}-Episode Moving Avg')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Total Reward', fontsize=12)
axes[0].set_title('Training Reward', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 中图：每回合步数
axes[1].plot(steps, alpha=0.3, color='#2ecc71', label='Raw Steps')
if len(steps) >= window:
    moving_steps = np.convolve(steps, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(steps)), moving_steps, 'r-', linewidth=2,
                 label=f'{window}-Episode Moving Avg')
axes[1].set_xlabel('Episode', fontsize=12)
axes[1].set_ylabel('Steps to Goal', fontsize=12)
axes[1].set_title('Steps per Episode', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# 右图：Q 表热力图（每个状态的最优Q值）
Q_max = np.max(Q_table, axis=1).reshape(env.rows, env.cols)
im = axes[2].imshow(Q_max, cmap='hot', interpolation='nearest')
axes[2].set_title('Q-Table Heatmap (Max Q per State)', fontsize=14)
axes[2].set_xlabel('Column', fontsize=12)
axes[2].set_ylabel('Row', fontsize=12)
fig.colorbar(im, ax=axes[2], label='Max Q Value')

plt.tight_layout()
plt.savefig('qlearning_training.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1800x500 with 3 Axes>

### 5.3 查看训练好的智能体走迷宫

In [6]:
# 使用训练好的Q表进行推理
def test_agent(env, Q, max_steps=100):
    """测试训练好的智能体"""
    state = env.reset()
    path = [state]
    total_reward = 0
    
    for step in range(max_steps):
        state_idx = env.state_to_idx(state)
        action = np.argmax(Q[state_idx])  # 纯利用，不探索
        new_state, reward, done = env.step(action)
        path.append(new_state)
        total_reward += reward
        state = new_state
        
        if done:
            break
    
    return path, total_reward, len(path) - 1

# 测试
path, reward, num_steps = test_agent(env, Q_table)
print(f"到达终点! 总步数: {num_steps}, 总奖励: {reward:.1f}")
print(f"路径: {path}")

# 可视化路径
fig, ax = plt.subplots(figsize=(7, 7))
draw_maze(env, agent_pos=path[-1], path=path, ax=ax)
ax.set_title(f'Q-Learning Agent Path ({num_steps} steps)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('qlearning_agent_path.png', dpi=150, bbox_inches='tight')
plt.show()

到达终点! 总步数: 11, 总奖励: 98.9
路径: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6)]


<Figure size 700x700 with 1 Axes>

In [7]:
# 可视化学到的策略 (每个状态的最佳动作)
fig, ax = plt.subplots(figsize=(8, 8))
draw_maze(env, ax=ax)

# 绘制每个状态的箭头（表示学到的策略）
arrow_chars = ['↑', '→', '↓', '←']
for r in range(env.rows):
    for c in range(env.cols):
        if env.maze[r, c] == 1:  # 跳过墙壁
            continue
        if (r, c) == tuple(env.goal):  # 跳过终点
            continue
        
        state_idx = r * env.cols + c
        best_action = np.argmax(Q_table[state_idx])
        
        # 绘制箭头
        dc, dr = env.ACTIONS[best_action]
        ax.annotate('', xy=(c + 0.5 + dc*0.3, env.rows-1-r + 0.5 - dr*0.3),
                    xytext=(c + 0.5, env.rows-1-r + 0.5),
                    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))

ax.set_title('Learned Policy (Arrows = Best Action)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('qlearning_policy.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 800x800 with 1 Axes>

## 🏋️ 练习题

### 练习1：概念理解

**Q1**: 解释马尔可夫性质的含义，为什么这个假设在强化学习中很重要？

**Q2**: 折扣因子 $\gamma$ 取不同的值（0、0.5、1.0）分别意味着什么？对 Q-Learning 的学习过程有什么影响？

**Q3**: 为什么 Q-Learning 被称为 "off-policy" 方法？它与 "on-policy" 方法（如 SARSA）有什么区别？

### 练习2：代码实践

**Q4**: 修改 Q-Learning 的超参数，观察对训练效果的影响：
- 将学习率 $\alpha$ 改为 0.01 和 0.5，比较训练曲线
- 将折扣因子 $\gamma$ 改为 0.5 和 0.99，比较训练曲线
- 移除 ε 衰减（固定 $\epsilon = 0.1$），观察效果

**Q5**: 设计一个更大的迷宫（如 10×10），观察 Q-Learning 需要多少回合才能学会。

In [8]:
# 练习4: 参考答案框架 - 超参数对比实验

configs = [
    {"name": "α=0.01, γ=0.95", "alpha": 0.01, "gamma": 0.95, "episodes": 500},
    {"name": "α=0.5, γ=0.95",  "alpha": 0.5,  "gamma": 0.95, "episodes": 500},
    {"name": "α=0.1, γ=0.5",  "alpha": 0.1,  "gamma": 0.5,  "episodes": 500},
    {"name": "α=0.1, γ=0.99", "alpha": 0.1,  "gamma": 0.99, "episodes": 500},
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, config in enumerate(configs):
    env_fresh = MazeEnvironment()
    _, rewards_i, _ = q_learning_train(
        env_fresh, episodes=config["episodes"],
        alpha=config["alpha"], gamma=config["gamma"],
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995
    )
    
    # 滑动平均
    window = 20
    if len(rewards_i) >= window:
        avg = np.convolve(rewards_i, np.ones(window)/window, mode='valid')
        axes[i].plot(avg, color='#e74c3c', linewidth=2)
    axes[i].set_title(config["name"], fontsize=13)
    axes[i].set_xlabel('Episode', fontsize=10)
    axes[i].set_ylabel('Reward', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Q-Learning Hyperparameter Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("分析:")
print("- α=0.01: 学习太慢，可能收敛不足")
print("- α=0.5: 学习快但不稳定，可能震荡")
print("- γ=0.5: 只看重近期奖励，策略可能短视")
print("- γ=0.99: 重视长远奖励，策略更有远见")

<Figure size 1400x1000 with 4 Axes>

分析:
- α=0.01: 学习太慢，可能收敛不足
- α=0.5: 学习快但不稳定，可能震荡
- γ=0.5: 只看重近期奖励，策略可能短视
- γ=0.99: 重视长远奖励，策略更有远见


---

## 📌 本节小结

| 知识点 | 要点 |
|--------|------|
| 强化学习框架 | Agent ↔ Environment (State, Action, Reward) |
| MDP | 用五元组 $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ 描述 |
| 价值函数 | $Q(s,a)$ 表示在状态 $s$ 采取动作 $a$ 的期望累积奖励 |
| 探索与利用 | ε-greedy: 以 $\epsilon$ 概率探索，$1-\epsilon$ 概率利用 |
| Q-Learning | 通过 TD 更新学习最优 Q 表 |
| Q-Learning 更新 | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'}Q(s',a') - Q(s,a)]$ |

**下一节**: [5.4 数据基础](./5.4_数据基础.ipynb)